---
title: RC4 and RC5 process parameters
description: Generate parser-checked robot control and welding source from explicit settings
output-file: application.process_parameters.html
---

In [ ]:
#| default_exp application.process_parameters

# RC4 and RC5 process parameters

This notebook is the source of `ribosome.application.process_parameters`. It turns explicit control and welding settings into a complete `NOP`–`END` instruction program, validates the matching parser, and returns a resource manifest.

Edit the exported cells here and run `python -m ribosome.prepare --profile llm-backend` from the Ribosome checkout to refresh the module. Exporting never runs the examples. The [step-by-step tutorial](06.tutorial.process_parameters.ipynb) uses the exported API; the [reference guide](../docs/process-parameters.md) documents all fields and limits.

The caller supplies taught positions, frames, controller register mappings, and process values. Generation checks syntax and the documented stud application limits. It does not calculate poses, optimize a welding recipe, execute a program, or establish physical safety.

## Dependencies and strict values

The runtime uses Pydantic 2, `treesitter4robotcode`, and `robot-language-lean-checker`. The backend checkout already installs these companion packages. Standalone users need the parser and checker Python packages available in the same environment; the compiled Lean executable is only needed for the separate execution/static-checking workflow.

Private aliases enforce finite numbers, integer indices, and unknown-field rejection. Boolean values are accepted only by the digital I/O models. These definitions are exported as implementation details.

In [ ]:
#| exporti
from __future__ import annotations
from decimal import Decimal
from typing import Annotated, Literal
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StrictFloat,
    StrictInt,
    field_validator,
    model_validator,
)
from robot_language_lean_checker.welding import diagnostics as welding_diagnostics
from treesitter4robotcode.dialects import normalize_dialect, parse_process_parameters

Number = StrictInt | Annotated[StrictFloat, Field(allow_inf_nan=False)]


Index = Annotated[int, Field(strict=True, ge=0)]


Name = Annotated[str, Field(pattern=r'^[A-Za-z_][A-Za-z0-9_]*$')]


Bank = Literal['I', 'R', 'P', 'PR', 'TF', 'UF', 'IN', 'OT', 'DI', 'DO']


class _Model(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True, frozen=True)

## Numeric bindings and welding settings

A named numeric value initializes an explicitly chosen `I` or `R` register. Integer registers require integer values. `WeldingParameters` groups positive current (A), voltage (V), wire feed (mm/s), and duration (s) bindings; the cell must already interpret the chosen registers in those units. No equipment mapping or recipe value is inferred.

Names can be referenced by later motion fields. Ordinary motion speed stays a runtime register read; other references are captured as literal values when generating source.

In [ ]:
#| export
class NumericParameter(_Model):
    """An explicit process value bound to an existing numeric register."""

    name: Name
    bank: Literal['I', 'R']
    index: Index
    value: Number

    @model_validator(mode='after')
    def integer_register_value(self):
        if self.bank == 'I' and type(self.value) is not int:
            raise ValueError('I registers require an integer parameter value')
        return self


class WeldingParameters(_Model):
    """Cell-mapped welding settings in explicit units, with no inferred recipe.

    Each register mapping must match the controller's welding interface. These
    assignments alone do not activate a gun or start a welding operation.
    """

    current_amperes: NumericParameter | None = None
    voltage_volts: NumericParameter | None = None
    wire_feed_mm_per_second: NumericParameter | None = None
    duration_seconds: NumericParameter | None = None

    @model_validator(mode='after')
    def positive_settings(self):
        parameters = self.bindings()
        if not parameters:
            raise ValueError('Welding parameters require at least one explicit setting')
        if any(parameter.value <= 0 for parameter in parameters):
            raise ValueError('Welding settings must be positive')
        return self

    def bindings(self) -> list[NumericParameter]:
        return [value for field in type(self).model_fields
                if (value := getattr(self, field)) is not None]


class ParameterRef(_Model):
    parameter: Name

In [ ]:
#| exporti
Scalar = Number | ParameterRef

## Ordered control and stud-welding steps

Common steps select taught positions with `MOVJ`/`MOVL`, write digital outputs, or wait for digital inputs. Each move requires speed, acceleration, and an explicit blend choice; `None` requests an exact stop. WAIT timeouts are finite positive values in the selected controller's native `T` units, without automatic conversion.

Native stud commands require local `P` targets and the RC5 application grammar. `STUDL` requires `ET`; `STUDJ` rejects it. The generator derives each gun's straight-line count, emits initialization, and finishes with `ProcessCheck`. Native RC4 stud requests are rejected; explicit register/I/O welding control remains available in either dialect.

In [ ]:
#| export
class Position(_Model):
    bank: Literal['P', 'PR'] = 'P'
    index: Index


class MotionStep(_Model):
    kind: Literal['move'] = 'move'
    motion: Literal['MOVJ', 'MOVL']
    target: Position
    speed: Scalar
    acceleration: Scalar
    # Required: None explicitly selects an exact stop; zero is a CNT value.
    blend: Scalar | None


class DigitalOutputStep(_Model):
    kind: Literal['output'] = 'output'
    channel: Index
    value: bool


class WaitInputStep(_Model):
    kind: Literal['wait_input'] = 'wait_input'
    channel: Index
    value: bool
    timeout: Scalar


class StudWeldStep(_Model):
    """SX322013-A2 native stud operation; supported by the RC5 parser only."""

    kind: Literal['stud_weld'] = 'stud_weld'
    motion: Literal['STUDJ', 'STUDL']
    target: Position
    speed: Scalar
    acceleration: Scalar
    blend: Scalar
    gun: Index
    program: Index
    et: Index | None = None
    interlock: Index | None = None

    @model_validator(mode='after')
    def supported_motion_fields(self):
        if self.target.bank != 'P':
            raise ValueError('Native stud commands require a local P target')
        if self.motion == 'STUDL' and self.et is None:
            raise ValueError('STUDL requires an explicit ET value')
        if self.motion == 'STUDJ' and self.et is not None:
            raise ValueError('STUDJ does not accept ET')
        return self

In [ ]:
#| exporti
Step = Annotated[MotionStep | DigitalOutputStep | WaitInputStep | StudWeldStep,
                 Field(discriminator='kind')]

## Process specification and resource bounds

An explicit dialect controls rendering and parsing. Duplicate names or register assignments are rejected, including duplicates across common and welding parameters. Bounds use actual controller bank names (`IN`/`OT` for RC4, `DI`/`DO` for RC5) and apply to every generated register, position, frame, and I/O reference. Missing bounds remain visible in the result.

Native stud operations also require `application_profile="SX322013-A2"`; settings from that revision are not applied to an unspecified application profile.

In [ ]:
#| export
class IndexRange(_Model):
    min: Index
    max: Index

    @model_validator(mode='after')
    def ordered(self):
        if self.min > self.max:
            raise ValueError('Register range min must not exceed max')
        return self


class ProcessSpec(_Model):
    dialect: Literal['rc4', 'rc5']
    parameters: list[NumericParameter] = Field(default_factory=list)
    welding: WeldingParameters | None = None
    application_profile: Literal['SX322013-A2'] | None = None
    tool_frame: Index | None = None
    user_frame: Index | None = None
    steps: list[Step] = Field(min_length=1)
    # Use actual bank names: RC4 IN/OT, RC5 DI/DO. Missing banks are reported.
    register_bounds: dict[Bank, IndexRange] = Field(default_factory=dict)

    @field_validator('dialect', mode='before')
    @classmethod
    def concrete_dialect(cls, value):
        return normalize_dialect(value)

    @model_validator(mode='after')
    def unique_bindings_and_dialect_banks(self):
        names, registers = set(), set()
        for parameter in self.bindings():
            if parameter.name in names:
                raise ValueError(f'Duplicate parameter name: {parameter.name}')
            address = (parameter.bank, parameter.index)
            if address in registers:
                raise ValueError(f'Duplicate parameter register: {address}')
            names.add(parameter.name)
            registers.add(address)
        other_banks = {'DI', 'DO'} if self.dialect == 'rc4' else {'IN', 'OT'}
        if other_banks.intersection(self.register_bounds):
            raise ValueError(f'Register bounds must use {self.dialect.upper()} bank names')
        if any(isinstance(step, StudWeldStep) for step in self.steps):
            if self.dialect != 'rc5':
                raise ValueError('Native stud welding is unsupported by the RC4 parser; use explicit cell register/I/O mappings')
            if self.application_profile != 'SX322013-A2':
                raise ValueError('Native stud welding requires application_profile="SX322013-A2"')
        return self

    def bindings(self) -> list[NumericParameter]:
        return self.parameters + (self.welding.bindings() if self.welding else [])

## Generation result

`GeneratedProcess` returns source, normalized dialect, initialized parameters, deduplicated resources, banks without configured bounds, parser validation, and documented application diagnostics. Syntax success is separate from an execution or physical-safety claim. `parsed_parameters` records the recognized process instructions and their named syntax-tree fields directly from the generated source. Each field retains its exact text, node type, and original-source position. Repeated field values remain lists, including repeated RC4 motion parameters. Register references stay unresolved; parsing a register assignment does not infer a welding quantity or unit. Numeric source uses fixed decimal notation so values normally printed with exponents remain parseable.

In [ ]:
#| exporti
class ParameterBinding(_Model):
    name: str
    bank: Literal['I', 'R']
    index: Index
    value: Number
    reference: str


class Resource(_Model):
    bank: Bank
    index: Index
    reference: str

In [ ]:
#| export
class GeneratedProcess(_Model):
    dialect: Literal['rc4', 'rc5']
    code: str
    parameters: list[ParameterBinding]
    resources: list[Resource]
    unbounded_banks: list[str]
    validation: dict
    parsed_parameters: dict = Field(default_factory=dict)
    application_profile: Literal['SX322013-A2'] | None = None
    domain_diagnostics: list[dict] = Field(default_factory=list)

In [ ]:
#| exporti
def _number(value: float) -> str:
    """Keep Python's round-trip value, without unsupported exponent notation."""
    return format(Decimal(str(value)), 'f')


def _reference(bank: str, index: int, dialect: str) -> str:
    if dialect == 'rc5':
        return f'{bank}[{index}]'
    if bank in {'IN', 'OT'}:
        return f'{bank}#{index}'
    if bank in {'TF', 'UF'}:
        return f'{bank} #{index}'
    return f'{bank}{index}'

## Deterministic dialect rendering

Generation revalidates the complete specification, initializes bindings, selects frames, and renders user steps in order. RC4 uses forms such as `MOVL P1 VL=50 ACC=80 CNT=0`; core RC5 uses `MOVL P[1] V = 50 ACC = 80 FINE`. The native RC5 stud application deliberately retains its documented `P1` and `VJ`/`VL` syntax.

The matching parser checks the completed source and extracts parameter records. A syntax error yields diagnostics and no partial instruction inventory. Records follow source order, including instructions nested in control flow; they do not describe an execution trace. Native stud parameters also pass through the existing application diagnostics, which enforce SX322013-A2 ranges without inventing upper limits for linear speed or interference-zone identifiers. Invalid specifications raise `ValueError`, including Pydantic `ValidationError`.

In [ ]:
#| export
def generate_process_parameters(spec: ProcessSpec | dict) -> GeneratedProcess:
    """Render a complete NOP..END program and an explicit resource manifest.

    Common motion speed parameters remain runtime register references. ACC/CNT,
    WAIT timeout, and native stud operands are resolved to literals at generation.
    Speeds/accelerations/timeouts must be positive and blend values nonnegative;
    controller-specific physical limits are left to the caller. Optional index
    bounds apply to every used resource, including taught positions and frames.
    Native RC5 stud operations require the documented application profile; gun
    counts are derived from the straight-line steps and checked at the end.
    Validation here checks generated syntax, not execution or physical safety.
    """
    # Revalidate model instances too: nested containers may have been mutated.
    spec = ProcessSpec.model_validate(spec.model_dump() if isinstance(spec, ProcessSpec) else spec)
    dialect = spec.dialect
    parameters = {parameter.name: parameter for parameter in spec.bindings()}
    resources: dict[tuple[str, int], Resource] = {}
    lines = ['NOP']

    def resource(bank, index):
        limits = spec.register_bounds.get(bank)
        if limits is not None and not limits.min <= index <= limits.max:
            raise ValueError(f'{bank} index {index} is outside configured range {limits.min}..{limits.max}')
        reference = _reference(bank, index, dialect)
        resources[(bank, index)] = Resource(bank=bank, index=index, reference=reference)
        return reference

    def scalar(value, field, *, allow_zero=False, register=False):
        binding = None
        if isinstance(value, ParameterRef):
            binding = parameters.get(value.parameter)
            if binding is None:
                raise ValueError(f'Unknown parameter {value.parameter!r} used by {field}')
            value = binding.value
        if value < 0 or (value == 0 and not allow_zero):
            constraint = 'nonnegative' if allow_zero else 'positive'
            raise ValueError(f'{field} must be {constraint}')
        return resource(binding.bank, binding.index) if register and binding else _number(value)

    bindings = []
    for parameter in spec.bindings():
        reference = resource(parameter.bank, parameter.index)
        lines.append(f'SET {reference} {_number(parameter.value)}')
        bindings.append(ParameterBinding(**parameter.model_dump(), reference=reference))

    for bank, index in [('TF', spec.tool_frame), ('UF', spec.user_frame)]:
        if index is not None:
            resource(bank, index)
            lines.append(f'SETFRAME {bank} {index}' if dialect == 'rc5' else f'{bank} #{index}')

    weld_counts: dict[int, int] = {}
    for step in spec.steps:
        if isinstance(step, StudWeldStep):
            weld_counts[step.gun] = weld_counts.get(step.gun, 0) + 1
    for gun, count in sorted(weld_counts.items()):
        lines.append(f'A20_STUD Init Gun={gun} Count={count}')

    for step in spec.steps:
        if isinstance(step, MotionStep):
            target = resource(step.target.bank, step.target.index)
            speed = scalar(step.speed, 'speed', register=True)
            acceleration = scalar(step.acceleration, 'acceleration')
            blend = None if step.blend is None else scalar(step.blend, 'blend', allow_zero=True)
            if dialect == 'rc5':
                termination = 'FINE' if blend is None else f'CNT = {blend}'
                lines.append(f'{step.motion} {target} V = {speed} ACC = {acceleration} {termination}')
            else:
                velocity = 'VJ' if step.motion == 'MOVJ' else 'VL'
                lines.append(f'{step.motion} {target} {velocity}={speed} ACC={acceleration} CNT={blend or "0"}')
        elif isinstance(step, DigitalOutputStep):
            channel = resource('DO' if dialect == 'rc5' else 'OT', step.channel)
            lines.append(f'OUT {channel} {int(step.value)}' if dialect == 'rc5'
                         else f'OUT {channel}={"ON" if step.value else "OFF"}')
        elif isinstance(step, WaitInputStep):
            channel = resource('DI' if dialect == 'rc5' else 'IN', step.channel)
            timeout = scalar(step.timeout, 'timeout')
            lines.append(f'WAIT {channel} = {int(step.value)} T = {timeout}' if dialect == 'rc5'
                         else f'WAIT {channel}={"ON" if step.value else "OFF"} T={timeout}')
        else:
            resource('P', step.target.index)
            speed = scalar(step.speed, 'welding speed')
            acceleration = scalar(step.acceleration, 'welding acceleration')
            blend = scalar(step.blend, 'welding blend', allow_zero=True)
            velocity = 'VJ' if step.motion == 'STUDJ' else 'VL'
            # The application grammar deliberately uses P1/VJ/VL even in RC5.
            line = f'{step.motion} P{step.target.index} {velocity}={speed} ACC={acceleration} CNT={blend}'
            if step.et is not None:
                line += f' ET={step.et}'
            if step.interlock is not None:
                line += f' INTERLOCK={step.interlock}'
            lines.append(line + f' GUN={step.gun} PROGNo={step.program}')

    if weld_counts:
        lines.append('A20_STUD ProcessCheck')
    lines.append('END')
    code = '\n'.join(lines)
    parsed_parameters = parse_process_parameters(code, dialect)
    validation = {key: parsed_parameters[key] for key in ('valid', 'diagnostics')}
    if not validation['valid']:
        raise ValueError(f'Generated {dialect.upper()} process failed syntax validation: {validation["diagnostics"]}')
    domain = welding_diagnostics(code, dialect, spec.application_profile) if weld_counts else []
    if any(finding['severity'] == 'error' for finding in domain):
        raise ValueError(f'Generated welding parameters violate the application profile: {domain}')
    return GeneratedProcess(
        dialect=dialect, code=code, parameters=bindings,
        resources=list(resources.values()),
        unbounded_banks=sorted({bank for bank, _ in resources} - spec.register_bounds.keys()),
        validation=validation, application_profile=spec.application_profile,
        domain_diagnostics=domain, parsed_parameters=parsed_parameters,
    )

## Small offline example

This example runs when you execute the notebook, and stays out of the exported module. See the [tutorial](06.tutorial.process_parameters.ipynb) for a full common-control and welding walkthrough.

In [ ]:
example = generate_process_parameters({
    'dialect': 'rc5',
    'steps': [{
        'kind': 'move', 'motion': 'MOVL', 'target': {'index': 1},
        'speed': 50, 'acceleration': 80, 'blend': None,
    }],
})
assert example.validation['valid']
assert example.unbounded_banks == ['P']
assert example.parsed_parameters['instructions'][0]['kind'] == 'motion'
print(example.parsed_parameters['instructions'][0]['fields'])
print(example.code)